# 06 · VAE condicional

Entrena un autoencoder variacional condicionado al régimen sobre el bloque conjunto, con rampa de beta y free bits para evitar el colapso posterior.

**Entradas**

- `data/processed/ventanas.npz`

**Salidas**

- `models/generadores/cvae/ (modelo.pkl o .keras, historial.csv, meta.json)`
- `data/synthetic/cvae.npz`
- `results/figures/kl_por_dimension_cvae.png`

**Tiempo estimado:** ~25 min en CPU (200 épocas sobre ~3.500 ventanas).

**Independencia.** Este notebook solo lee `data/processed/ventanas.npz` (notebook 02) y solo escribe en `models/generadores/cvae/` y `data/synthetic/cvae.npz`. No depende de ningún otro notebook de generador ni de sus salidas, de modo que los notebooks 04 a 10 pueden ejecutarse en paralelo y en cualquier orden por distintas personas.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import regimenes
from src.generadores import base

v = config.ventanas()
n_regimenes = config.n_regimenes()

bloque_train = ventanas.empaquetar(train)
print("bloque de train:", bloque_train.shape, "· d esperada:", ventanas.dimension_bloque(v))
regimenes.distribucion(train.y_reg, n_regimenes)

## Por qué un VAE condicional

El decoder reconstruye únicamente el bloque `[X ‖ y_vol]`; el régimen no forma
parte de lo reconstruido porque es la condición. La consecuencia práctica es que la
etiqueta de un dato sintético es **exacta**: es la `c` impuesta al muestrear, no
una salida continua redondeada. Eso es lo que permite pedir "500 muestras de
crisis" en vez de generar mucho y filtrar.

Dos decisiones que condicionan el resultado:

**Rampa de beta.** Con el KL a peso completo desde la época 0, el modelo minimiza
la divergencia apagando dimensiones latentes antes de que el decoder aprenda a
usarlas: colapso posterior. La rampa lineal deja que la reconstrucción se
establezca primero.

**Free bits.** Un suelo por dimensión latente impide que el KL de una dimensión
concreta caiga a cero. Sin él, un VAE sobre datos estandarizados de dimensión alta
tiende a usar tres o cuatro dimensiones de las 32 disponibles.

La salida del decoder es **lineal**, sin sigmoid ni tanh. No es una elección de
estilo: la verosimilitud asumida es gaussiana, luego el soporte es todo `R`.

In [ ]:
bloque_val = ventanas.empaquetar(val)

generador = base.instanciar(
    "cvae",
    n_regimenes=n_regimenes,
    dim_latente=32,
    unidades_ocultas=512,
    beta=2.0,
    epocas_rampa_beta=30,
    free_bits=0.05,
    epocas=200,
    tam_lote=128,
    verboso=25,
)
generador.fit(bloque_train, train.y_reg, bloque_val=bloque_val, y_reg_val=val.y_reg)
generador

## Convergencia

Tres curvas que hay que leer juntas:

- **pérdida de reconstrucción**: debe bajar y aplanarse;
- **KL**: sube durante la rampa de beta y luego se estabiliza. Si cae a cero, el
  modelo ha colapsado y el decoder ignora el latente;
- **unidades activas**: cuántas de las 32 dimensiones latentes transportan
  información. Es el diagnóstico que dice si el cuello de botella está bien
  dimensionado.

Las curvas de validación son las que revelan el sobreajuste del decoder: si la
reconstrucción de train sigue bajando mientras la de validación sube, el modelo
está memorizando ventanas de entrenamiento.

In [ ]:
fig, eje = plt.subplots()
viz.curva_convergencia(generador.historial, "Convergencia · " + generador.etiqueta, eje=eje)
eje.set_yscale("log")
viz.guardar(fig, "convergencia_cvae")

generador.historial.tail(3).round(3)

## Uso del espacio latente

El KL por dimensión y por época dice qué dimensiones aprende a usar el modelo y
cuándo. Una franja horizontal oscura es una dimensión activa; el resto son
dimensiones apagadas que el decoder ignora. Si casi todas están apagadas, sobra
cuello de botella o falta rampa.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 4))

imagen = eje.imshow(
    generador.kl_por_dimension.T, aspect="auto", cmap="Blues", origin="lower"
)
eje.set_xlabel("época")
eje.set_ylabel("dimensión latente")
eje.set_title("KL por dimensión latente (nats)")
eje.grid(False)
fig.colorbar(imagen, ax=eje, fraction=0.046, label="nats")
viz.guardar(fig, "kl_por_dimension_cvae")

activas = (generador.kl_por_dimension[-1] > generador.umbral_unidad_activa).sum()
print("Dimensiones activas al final:", int(activas), "de", generador.dim_latente)

## Inspección visual

Proyección PCA de reales y sintéticos, con la PCA ajustada **solo con los reales**
para que los ejes describan la estructura del mercado y no la del generador.

Es la comprobación más rápida y la que detecta los dos fallos gruesos: si la nube
sintética no cubre la real, el generador ha colapsado a un modo; si la desborda
ampliamente, está inventando configuraciones de mercado que nunca ocurrieron.

Se mira el régimen de crisis porque es el que tiene menos datos reales y, por
tanto, donde el generador tiene más margen para desviarse.

In [ ]:
CRISIS = n_regimenes - 1

muestra_crisis = generador.generate(600, regimen=CRISIS)
reales_crisis = bloque_train[train.y_reg == CRISIS]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
viz.real_vs_sintetico(bloque_train, generador.generate(600, regimen=0),
                      "{} · régimen de calma".format(generador.etiqueta), eje=ejes[0])
viz.real_vs_sintetico(reales_crisis, muestra_crisis,
                      "{} · régimen de crisis".format(generador.etiqueta), eje=ejes[1])
fig.tight_layout()
viz.guardar(fig, "pca_" + generador.nombre)

print("reales de crisis:", len(reales_crisis), "· sintéticos generados:", len(muestra_crisis))

## Banco de muestras

Se genera un banco uniforme por régimen y se exporta a `data/synthetic/`. La mezcla
concreta de cada dataset la decide el notebook 11 muestreando de este banco, no
volviendo a invocar al generador: así el barrido no necesita tener los siete
modelos cargados en memoria y dos ejecuciones del notebook 12 usan exactamente las
mismas muestras sintéticas.

El banco es uniforme —no replica el desbalance real— porque la política de reparto
es un grado de libertad del experimento y se aplica después.

In [ ]:
MUESTRAS_POR_REGIMEN = 3000

reparto = {k: MUESTRAS_POR_REGIMEN for k in range(n_regimenes)}
bloques_sint, y_sint = generador.generate_dataset(reparto)

print("banco:", bloques_sint.shape, "· etiquetas:", np.bincount(y_sint, minlength=n_regimenes))
print("rango de valores:", round(float(bloques_sint.min()), 2), "→",
      round(float(bloques_sint.max()), 2),
      "(referencia real:", round(float(bloque_train.min()), 2), "→",
      round(float(bloque_train.max()), 2), ")")

## Persistencia

`guardar()` deja el modelo, la curva de convergencia y los metadatos en
`models/generadores/`. Es lo que permite que el resto del grupo salte directamente
al análisis sin reentrenar nada.

In [ ]:
ruta_muestras = generador.exportar_muestras(bloques_sint, y_sint)
ruta_modelo = generador.guardar()

print("muestras:", ruta_muestras)
print("modelo:  ", ruta_modelo)
pd.Series(generador.resumen_convergencia()).round(4)

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_MODELOS_GEN / "cvae" / "meta.json",
    src.DIR_MODELOS_GEN / "cvae" / "historial.csv",
    src.DIR_SINTETICO / "cvae.npz",
    src.DIR_FIGURAS / "convergencia_cvae.png",
    src.DIR_FIGURAS / "pca_cvae.png",
    src.DIR_FIGURAS / "kl_por_dimension_cvae.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
